In [30]:
import pandas as pd
census = pd.read_csv("/census-pca-demography-districts.csv")
malls = pd.read_excel("/mall_counts_raw.xlsx")

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [32]:
nsdp = pd.read_excel("/nsdp_per_capita_raw.xlsx", header=6)
nsdp = nsdp.iloc[:, [1, 2]]
nsdp.columns = ['state_name', 'nsdp_per_capita']
nsdp = nsdp.dropna(subset=['state_name'])
nsdp['state_name'] = nsdp['state_name'].str.strip()

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [33]:
malls.columns = malls.columns.str.strip()
malls['City'] = malls['City'].str.strip().str.title()
malls['State'] = malls['State'].str.strip()

In [34]:
cities = ["Lucknow", "Indore", "Surat", "Nagpur", "Coimbatore", "Jaipur"]
df = census[census['district_name'].str.strip().str.title().isin(cities)]

In [35]:
urban_pop = df[df['rural_urban']=='Urban'].groupby('district_name')['population'].sum()
total_pop = df.groupby('district_name')['population'].sum()
urbanization_pct = (urban_pop / total_pop * 100).round(2)

In [36]:
city_state = {"Lucknow":"Uttar Pradesh","Indore":"Madhya Pradesh","Surat":"Gujarat",
              "Nagpur":"Maharashtra","Coimbatore":"Tamil Nadu","Jaipur":"Rajasthan"}

In [37]:
final = df.merge(malls[['City','Mall Count']], left_on='district_name', right_on='City')
final = final.groupby('City').agg(total_pop=('population','sum'), mall_count=('Mall Count','first')).reset_index()
final['malls_per_capita'] = final['mall_count'] / final['total_pop'] * 100000

In [38]:
final['urbanization_pct'] = final['City'].map(urbanization_pct)

In [39]:
final['population'] = final['total_pop']

In [40]:
final['state_name'] = final['City'].map(city_state)

In [41]:
nsdp_map = nsdp.set_index('state_name')['nsdp_per_capita']
final['nsdp_per_capita'] = final['state_name'].map(nsdp_map)

In [42]:
print(final[['City','state_name','nsdp_per_capita','population','urbanization_pct','malls_per_capita']])

         City      state_name nsdp_per_capita  population  urbanization_pct  \
0  Coimbatore      Tamil Nadu          361619    90184220             85.64   
1      Indore  Madhya Pradesh          152615    54054328             76.40   
2      Jaipur       Rajasthan          185053   113057840             58.52   
3     Lucknow   Uttar Pradesh          108572   108656100             79.58   
4      Nagpur     Maharashtra          309340   113581096             80.18   
5       Surat         Gujarat               -   160374740             88.38   

   malls_per_capita  
0          0.003327  
1          0.011100  
2          0.008845  
3          0.011044  
4          0.004402  
5          0.003118  


In [44]:
final['nsdp_per_capita'] = final['nsdp_per_capita'].replace('-', pd.NA)
final['nsdp_per_capita'] = pd.to_numeric(final['nsdp_per_capita'], errors='coerce')
print(final[['City','nsdp_per_capita']])

         City  nsdp_per_capita
0  Coimbatore         361619.0
1      Indore         152615.0
2      Jaipur         185053.0
3     Lucknow         108572.0
4      Nagpur         309340.0
5       Surat              NaN


In [45]:
print(df[df['district_name']=='Coimbatore'][['level','age','gender','social_group','population']].head(20))

         level           age  gender social_group  population
5676   Village  0_To_6_Years    Male          NaN       35629
5677   Village           NaN  Female          NaN         212
5678   Village           NaN    Male          NaN        1254
5679   Village           NaN   Total          NaN        2710
5680   Village           NaN    Male          NaN      113894
5681   Village           NaN    Male          NaN      137692
5682   Village           NaN   Total          NaN      140577
5683   Village           NaN    Male          NaN      305821
5684   Village           NaN  Female          NaN       21876
5685   Village  0_To_6_Years  Female          NaN       34509
5686   Village           NaN    Male          NaN        9797
5687   Village           NaN    Male           Sc       93691
12780     Town           NaN    Male          NaN     1089072
12781     Ward           NaN    Male          NaN      738698
12782     Ward           NaN    Male          NaN      504951
12783   

In [47]:
nsdp_check = pd.read_excel("/nsdp_per_capita_raw.xlsx", header=6)
print(nsdp_check[nsdp_check.iloc[:,1].str.strip()=='Gujarat'])

   Unnamed: 0 Unnamed: 1 2024-25    2023-24     2022-23    2021-22     \
9         NaN    Gujarat          -          -    272451.0     241584   

  2020-21    2019-20    2018-19    2017-18    2016-17    2015-16     \
9     207324     212428     197457     176961     156295     139254   

  2014-15    2013-14    2012-13    2011-12     
9     127017     113139     102826      87481  


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [48]:
final.loc[final['City']=='Surat', 'nsdp_per_capita'] = 272451

In [49]:
final[['City','nsdp_per_capita']]

,City,nsdp_per_capita
0,Coimbatore,361619.0
1,Indore,152615.0
2,Jaipur,185053.0
3,Lucknow,108572.0
4,Nagpur,309340.0
5,Surat,272451.0


In [50]:
print(df[df['district_name']=='Coimbatore'][['level','age','gender','social_group','population']].head(20))

         level           age  gender social_group  population
5676   Village  0_To_6_Years    Male          NaN       35629
5677   Village           NaN  Female          NaN         212
5678   Village           NaN    Male          NaN        1254
5679   Village           NaN   Total          NaN        2710
5680   Village           NaN    Male          NaN      113894
5681   Village           NaN    Male          NaN      137692
5682   Village           NaN   Total          NaN      140577
5683   Village           NaN    Male          NaN      305821
5684   Village           NaN  Female          NaN       21876
5685   Village  0_To_6_Years  Female          NaN       34509
5686   Village           NaN    Male          NaN        9797
5687   Village           NaN    Male           Sc       93691
12780     Town           NaN    Male          NaN     1089072
12781     Ward           NaN    Male          NaN      738698
12782     Ward           NaN    Male          NaN      504951
12783   

In [51]:
print(df['level'].unique())

['Village' 'Ward' 'Town']


In [52]:
clean = census[
    (census['district_name'].str.strip().str.title().isin(cities)) &
    (census['level']=='District') &        # adjust based on what you see above
    (census['age']=='Total') &
    (census['gender']=='Total') &
    (census['social_group']=='Total') &
    (census['worker_type']=='Total') &
    (census['occupation']=='Total')
]
print(clean[['district_name','population']])

Empty DataFrame
Columns: [district_name, population]
Index: []


In [53]:
print(df['age'].unique())
print(df['gender'].unique())
print(df['social_group'].unique())
print(df['worker_type'].unique())
print(df['occupation'].unique())

[nan '0_To_6_Years']
['Female' 'Male' 'Total']
[nan 'Total' 'St' 'Sc' 'General']
['Main' 'Marg_3_6_' nan 'Marg_0_3_' 'Total']
['Total' 'Other' nan 'Agricultural_Labourers' 'Household_Industry_Worker'
 'Cultivator']


In [54]:
clean = census[
    (census['district_name'].str.strip().str.title().isin(cities)) &
    (census['age'].isna()) &
    (census['gender']=='Total') &
    (census['social_group']=='Total') &
    (census['worker_type']=='Total') &
    (census['occupation']=='Total')
]
print(clean[['district_name','level','population']])

Empty DataFrame
Columns: [district_name, level, population]
Index: []


In [55]:
step1 = census[census['district_name'].str.strip().str.title().isin(cities)]
print(len(step1))

step2 = step1[step1['age'].isna()]
print(len(step2))

step3 = step2[step2['gender']=='Total']
print(len(step3))

step4 = step3[step3['social_group']=='Total']
print(len(step4))

step5 = step4[step4['worker_type']=='Total']
print(len(step5))

step6 = step5[step5['occupation']=='Total']
print(len(step6))

1620
1566
522
18
0
0


In [56]:
print(step4['worker_type'].unique())
print(step4['worker_type'].apply(repr).unique())  # shows hidden whitespace

[nan]
['nan']


In [57]:
step5 = step4[step4['worker_type'].isna()]
print(len(step5))

step6 = step5[step5['occupation']=='Total']
print(len(step6))

print(step6[['district_name','level','population']])

18
0
Empty DataFrame
Columns: [district_name, level, population]
Index: []


In [58]:
print(step5['occupation'].unique())

[nan]


In [59]:
step6 = step5[step5['occupation'].isna()]
print(len(step6))
print(step6[['district_name','level','population']])

18
       district_name    level  population
3192          Indore  Village      848988
10469         Nagpur     Ward     3178759
25900         Indore     Ward     2427709
26949          Surat     Town     4815400
46106     Coimbatore     Town     2618940
55631     Coimbatore  Village      840531
60407         Nagpur     Town     3178759
70515         Nagpur  Village     1474811
92360         Indore     Town      435287
93468          Surat     Ward     4849213
112532    Coimbatore     Ward     2618940
120144         Surat  Village     1232109
137317        Jaipur     Town     1189258
138272       Lucknow     Ward     3038996
151144        Jaipur     Ward     3471847
152147       Lucknow     Town     3038996
155374        Jaipur  Village     3154331
162887       Lucknow  Village     1550842


In [61]:
final_pop = step6[step6['level'].isin(['Village','Town'])].groupby('district_name')['population'].sum()
print(final_pop)

district_name
Coimbatore    3459471
Indore        1284275
Jaipur        4343589
Lucknow       4589838
Nagpur        4653570
Surat         6047509
Name: population, dtype: int64


In [62]:
urban_pop = step6[step6['level']=='Town'].groupby('district_name')['population'].sum()
urbanization_pct = (urban_pop / final_pop * 100).round(2)
print(urbanization_pct)

district_name
Coimbatore    75.70
Indore        33.89
Jaipur        27.38
Lucknow       66.21
Nagpur        68.31
Surat         79.63
Name: population, dtype: float64


In [63]:
final = pd.DataFrame({
    'City': final_pop.index,
    'population': final_pop.values
})
final['urbanization_pct'] = final['City'].map(urbanization_pct)

In [64]:
final = final.merge(malls[['City','Mall Count']], on='City')
final['malls_per_capita'] = final['Mall Count'] / final['population'] * 100000

In [65]:
final['state_name'] = final['City'].map(city_state)
nsdp_map = nsdp.set_index('state_name')['nsdp_per_capita']
final['nsdp_per_capita'] = final['state_name'].map(nsdp_map)
final.loc[final['City']=='Surat', 'nsdp_per_capita'] = 272451

In [66]:
print(final)

         City  population  urbanization_pct  Mall Count  malls_per_capita  \
0  Coimbatore     3459471             75.70           3          0.086718   
1      Indore     1284275             33.89           6          0.467190   
2      Jaipur     4343589             27.38          10          0.230224   
3     Lucknow     4589838             66.21          12          0.261447   
4      Nagpur     4653570             68.31           5          0.107444   
5       Surat     6047509             79.63           5          0.082679   

       state_name nsdp_per_capita  
0      Tamil Nadu          361619  
1  Madhya Pradesh          152615  
2       Rajasthan          185053  
3   Uttar Pradesh          108572  
4     Maharashtra          309340  
5         Gujarat          272451  


In [67]:
for col in ['population','urbanization_pct','nsdp_per_capita']:
    final[col+'_score'] = (final[col]-final[col].min())/(final[col].max()-final[col].min())*100

final['saturation_score'] = 100 - (final['malls_per_capita']-final['malls_per_capita'].min())/(final['malls_per_capita'].max()-final['malls_per_capita'].min())*100

print(final[['City','population_score','urbanization_pct_score','nsdp_per_capita_score','saturation_score']])

         City  population_score  urbanization_pct_score nsdp_per_capita_score  \
0  Coimbatore         45.666369               92.478469                 100.0   
1      Indore          0.000000               12.459330             17.405067   
2      Jaipur         64.227665                0.000000              30.22403   
3     Lucknow         69.397451               74.315789                   0.0   
4      Nagpur         70.735450               78.334928             79.340202   
5       Surat        100.000000              100.000000             64.762277   

   saturation_score  
0         98.949369  
1          0.000000  
2         61.627705  
3         53.507579  
4         93.559164  
5        100.000000  


In [68]:
final['nsdp_per_capita'] = pd.to_numeric(final['nsdp_per_capita'], errors='coerce')
final['nsdp_per_capita_score'] = (final['nsdp_per_capita']-final['nsdp_per_capita'].min())/(final['nsdp_per_capita'].max()-final['nsdp_per_capita'].min())*100
print(final[['City','nsdp_per_capita_score']])

         City  nsdp_per_capita_score
0  Coimbatore             100.000000
1      Indore              17.405067
2      Jaipur              30.224030
3     Lucknow               0.000000
4      Nagpur              79.340202
5       Surat              64.762277


In [71]:
from google.colab import files
files.download("master_scoring_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>